# Exploration → leaderboard → final selection

The pattern an agent uses to search for a good model: run **8–10 explorations**, let each one record
itself, then build a **leaderboard from the ledger** (not from notebook memory), export it as CSV,
inspect the top candidate's full result file, and **promote the winner into the catalog**.

The key idea: you do not have to track results yourself. Every `run_recipe` writes three things —
a `results_file` (the full payload), a **run record** (`list_runs` / `get_run`, carrying the whole
recipe + score), and a **typed result** (`list_results` / `get_result`, carrying model + metrics +
the file pointer). So after N explorations, the leaderboard is just a query over what the server
already stored.

Runs entirely on classical sktime — no torch, no network.

## Setup

In [ ]:
import os, sys, json, itertools, warnings
warnings.filterwarnings("ignore")
SRC = os.path.abspath("src")
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TSFM_STORE"] = "memory"
sys.path.insert(0, SRC)

import numpy as np, pandas as pd
from servers.tsfm import main as M
from servers.tsfm.io import refs

def dump(r): return r.model_dump() if hasattr(r, "model_dump") else r

SP, N = 24, 336
rng = np.random.RandomState(3)
t = np.arange(N)
load = 22 + 6*np.sin(t/SP*2*np.pi) + 2*np.sin(t/168*2*np.pi) + 0.01*t + rng.normal(0, .4, N)
ref = refs.materialize_iot(load, asset_id="chiller_8")
print("series:", N, "points | store:", type(M._STORE).__name__)

## 1. The exploration space

Ten candidates: different model families **and** different hyperparameters on the same family — the
mix an agent actually sweeps. Each is a catalog card so `run_recipe` can serve it by `model_id`.

In [ ]:
EXPLORATIONS = {
    "naive_last":       ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "last"}),
    "naive_seasonal":   ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "last", "sp": SP}),
    "naive_mean":       ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "mean", "sp": SP}),
    "theta_sp24":       ("sktime.forecasting.theta.ThetaForecaster", {"sp": SP}),
    "theta_sp12":       ("sktime.forecasting.theta.ThetaForecaster", {"sp": 12}),
    "autoreg_l24":      ("sktime.forecasting.auto_reg.AutoREG", {"lags": 24}),
    "autoreg_l48":      ("sktime.forecasting.auto_reg.AutoREG", {"lags": 48}),
    "autoreg_l12":      ("sktime.forecasting.auto_reg.AutoREG", {"lags": 12}),
    "expsmoothing":     ("sktime.forecasting.exp_smoothing.ExponentialSmoothing",
                         {"sp": SP, "seasonal": "add", "trend": "add"}),
    "trend_poly":       ("sktime.forecasting.trend.PolynomialTrendForecaster", {"degree": 2}),
}
for mid, (cls, params) in EXPLORATIONS.items():
    M.register_model({"model_id": mid, "description": f"exploration candidate {mid}",
                      "task_ids": ["tsfm_forecasting"], "provenance": "trained", "domain": "energy",
                      "sktime_class": cls, "params": params, "tags": ["sweep"]})
print(len(EXPLORATIONS), "candidates registered")

## 2. Run the explorations

Ten `run_recipe` calls, same series and horizon, only the `model_id` changing. We deliberately
**do not** collect the scores here — each run records itself. We just fire them.

In [ ]:
FH = [1, 2, 3, 4, 5, 6]
for mid in EXPLORATIONS:
    r = dump(M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                          asset_id="chiller_8",
                          recipe={"estimator": {"model_id": mid}, "fh": FH,
                                  "eval": {"metrics": ["mape"]}}))
    tag = "ok" if "error" not in r else f"ERROR {r['error'][:40]}"
    print(f"  ran {mid:16s} -> {tag}")
print("\nall explorations done — nothing tracked by hand; the ledger has it")

## 3. Build the leaderboard FROM THE LEDGER

Now query what the server stored. `list_results` gives one record per exploration, each carrying the
`model_id`, its `metrics` (`score`, `folds`), and the `results_file` pointer. The params that distinguish candidates (e.g. `lags=24` vs `48`) live on the catalog **card**, so we
pull them from `find_models` - a run recorded by `model_id` stores only the id.

In [ ]:
# every persisted forecasting result for this asset - this is the leaderboard source
results = dump(M.list_results(task_type="tsfm_forecasting", asset_id="chiller_8"))["results"]

# the params that distinguish candidates live on the catalog CARD (a run by model_id records only
# the id), so pull them from list_models, which returns EVERY card with its params.
# (use list_models, not find_models — find_models defaults to top_k=5 and would miss candidates.)
params_of = {m["model_id"]: m.get("params", {})
             for m in dump(M.list_models())["models"]}

rows = []
for res in results:
    m = (res.get("metrics") or [{}])[0]
    rows.append({"model_id": res["model_id"],
                 "MAPE": m.get("score"),
                 "folds": m.get("folds"),
                 "params": params_of.get(res["model_id"], {}),
                 "result_id": res["result_id"],
                 "results_file": res["results_file"]})

leaderboard = (pd.DataFrame(rows)
               .dropna(subset=["MAPE"])
               .sort_values("MAPE")
               .reset_index(drop=True))
leaderboard.index = leaderboard.index + 1              # rank starting at 1
display(leaderboard[["model_id", "MAPE", "folds", "params"]])

## 4. Export the leaderboard as CSV

In [ ]:
csv_path = os.path.abspath("chiller_8_leaderboard.csv")
leaderboard[["model_id", "MAPE", "folds", "params", "result_id", "results_file"]].to_csv(csv_path, index_label="rank")
print("leaderboard written to:", csv_path)
print(pd.read_csv(csv_path).head(4).to_string(index=False))

## 5. Inspect the top candidate's full result

The leaderboard is a summary; the winner's `results_file` holds the full payload (forecast head,
intervals, audit). Open it via the pointer the ledger recorded — no need to have kept it.

In [ ]:
top = leaderboard.iloc[0]
print(f"top candidate: {top.model_id}  MAPE {top.MAPE:.4f}  params {top.params}")

# open the winner's payload straight from the recorded pointer
payload = json.loads(open(top.results_file[7:]).read())
print("\nresult payload keys:", list(payload.keys()))
print("forecast_head       :", payload.get("forecast_head"))
print("metric / score      :", payload.get("metric"), payload.get("backtest_score"))

# get_result gives the same record through the tool surface
gr = dump(M.get_result(task_type="tsfm_forecasting", result_id=top.result_id))
print("\nget_result round-trips:", gr["results_file"] == top.results_file)

## 6. Select the final: promote the winner, retire the rest

The decision goes back into the catalog. The winner is tagged `recommended` with its evidence; the
others are deprecated **with their leaderboard score in the reason**, so a later agent inherits the
ranking instead of re-running the sweep.

In [ ]:
WINNER = top.model_id
M.update_model(WINNER, {"tags": ["forecast", "recommended"],
    "description": f"SELECTED for chiller-8 after a 10-way sweep: MAPE {top.MAPE:.4f} "
                   f"over {int(top.folds)} folds. params={top.params}."})

for _, row in leaderboard.iloc[1:].iterrows():
    M.deprecate_model(row.model_id,
        reason=f"chiller-8 sweep rank {row.name}: MAPE {row.MAPE:.4f} vs {top.MAPE:.4f} for {WINNER}")

live = [m["model_id"] for m in dump(M.find_models(task_id="tsfm_forecasting"))["models"]]
n_runs = len(dump(M.list_runs(asset_id="chiller_8"))["runs"])
print("promoted:", WINNER)
print("active forecasting models now:", live)
print("\nthe sweep is now reproducible from the store: leaderboard CSV +",
      f"{n_runs} run records + {len(results)} result records.")

## How this is possible — the three-legged record

You never had to keep a results table in the notebook. Each `run_recipe` persisted:

| leg | tool to read it | what it carries | durable? |
|---|---|---|---|
| results file | open the `results_file` pointer | full payload: forecast head, intervals, audit | file in `TSFM_WORKDIR` |
| run record | `list_runs` / `get_run` | the **full recipe** (model + params) + score + folds | CouchDB (default backend) |
| typed result | `list_results` / `get_result` | model_id + metrics + the file pointer | CouchDB |

So the leaderboard is a query, not bookkeeping:

1. **explore** — fire N `run_recipe` calls; each self-records
2. **rank** — `list_results(task_type, asset_id)`, sort by `metrics.score`; join `find_models` for each card's params
3. **export** — the DataFrame → CSV
4. **inspect** — open the winner's `results_file` (or `get_result`) for the detail
5. **select** — `update_model` the winner, `deprecate_model` the rest with their scores

Over CouchDB (drop `TSFM_STORE=memory`) the run and result records survive the session, so the
leaderboard is rebuildable days later from `list_results` alone — even across different notebooks or
agents working the same asset.